# <a id='toc1_'></a>[Section 2: Preprocessing - Feature Engineering, Data Preparation](#toc0_)

The goal of this notebook was to take all the discoveries from my initial exploration of the data and make the necessary feature engineering with the goal of getting the data to a model-ready state. I will address the customer_type by combining the two transient customer types and removing both Group and Contract customer types. I feature engineered a variable for length of stay which I thought would be interesting to explore if longer or shorter stay patterns influence cancellations or not. 

Another step included in this notebook was an iterative step, that due to overfitting in my initial modeling, I returned to feature engineering and made a couple adjustments. I discovered that the column “reservation_status” caused leakage. 


**Table of contents**<a id='toc0_'></a>    
- [Section 2: Preprocessing - Feature Engineering, Data Preparation](#toc1_)    
- [1. Set Up](#toc2_)    
- [2. Feature Engineering](#toc3_)    
  - [2.1 Customer Type Feature Engineering](#toc3_1_)    
  - [2.2 Length of Stay Feature Engineering](#toc3_2_)    
  - [2.3 Remove reservation_status](#toc3_3_)    
  - [2.4 Remove highly correlated features: previous_cancellations](#toc3_4_)    
  - [2.5 Extract features from datetime column: arrival_date](#toc3_5_)    
  - [2.6 One Hot Encoding Feature Engineering:](#toc3_6_)    
- [3. Summary](#toc4_)    
- [4. Data Export](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc2_'></a>[1. Set Up](#toc0_)

In [ ]:
# Standard imports
import numpy as np
import pandas as pd

# Plotting
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go

# Modeling
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso, Ridge, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

#Metrics 
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay
import shap
from sklearn.inspection import permutation_importance
import lime
import lime.lime_tabular

# Balancing
from imblearn.over_sampling import SMOTE


from tempfile import mkdtemp
import joblib

In [135]:
# show all dataframe columns
pd.set_option('display.max_columns', None)
# set matplotlib global settings eg. figsize
plt.rcParams['figure.figsize'] = (8.0, 6.0)

In [136]:
# Import hotel_engineering_df
clean_df = pd.read_csv('../data/processed/clean_df.csv')
clean_df.shape

(119390, 28)

In [137]:
clean_df.head(5)

,hotel,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,day_demand
0,Resort Hotel,0,342,27,0,0,2,0,0,1,Direct,Direct,0,0,0,3,No Deposit,0,0,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,122
1,Resort Hotel,0,737,27,0,0,2,0,0,1,Direct,Direct,0,0,0,4,No Deposit,0,0,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,122
2,Resort Hotel,0,7,27,0,1,1,0,0,1,Direct,Direct,0,0,0,0,No Deposit,0,0,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,122
3,Resort Hotel,0,13,27,0,1,1,0,0,1,Corporate,Corporate,0,0,0,0,No Deposit,1,0,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,122
4,Resort Hotel,0,14,27,0,2,2,0,0,1,Online TA,TA/TO,0,0,0,0,No Deposit,1,0,0,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,122


In [138]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 28 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_week_number        119390 non-null  int64  
 4   stays_in_weekend_nights         119390 non-null  int64  
 5   stays_in_week_nights            119390 non-null  int64  
 6   adults                          119390 non-null  int64  
 7   children                        119390 non-null  int64  
 8   babies                          119390 non-null  int64  
 9   meal                            119390 non-null  int64  
 10  market_segment                  119390 non-null  object 
 11  distribution_channel            119390 non-null  object 
 12  is_repeated_gues

In [139]:
#Converting date columns back to datetime because the import converted it to an object
clean_df['arrival_date'] = pd.to_datetime(clean_df['arrival_date'])
clean_df['reservation_status_date'] = pd.to_datetime(clean_df['reservation_status_date'])

In [ ]:
# Create a copy of the dataframe for engineering
engineering_df = clean_df.copy()

# <a id='toc3_'></a>[2. Feature Engineering](#toc0_)
1. Group `customer_type` Transient and Transient Party together
2. Remove Group and Contract `customer_type`'s to isolate only Transient
3. Calculate and group Length of Stay into buckets
4. Remove `reservation_status` feature
5. Extract features from datetime column `arrival_date`
6. One Hot Encoding for remaining categorical features

## <a id='toc3_1_'></a>[2.1 Customer Type Feature Engineering](#toc0_)
Looking at customer types, I will combine the two Transient customer_types and remove reservations under the Group and Contract customer types since there is very few group reservations in this dataset.  Also, with my experience with of Group and Contract demand, they are very different and much harder to predict compared to transient demand. Therefore I will focus primarily on transient demand only.

- Group Customer_Type Transient and Transient Party together
- Isolate only Transient customer_type by removing Group and Contract customer_types 

In [ ]:
# Checking customer_type values
engineering_df['customer_type'].value_counts()

customer_type
Transient          89613
Transient-Party    25124
Contract            4076
Group                577
Name: count, dtype: int64

In [142]:
# Change all Transient-Party to be Transient
engineering_df['customer_type'].replace('Transient-Party', 'Transient',inplace=True)

In [143]:
# Sanity Check
engineering_df['customer_type'].value_counts()

customer_type
Transient    114737
Contract       4076
Group           577
Name: count, dtype: int64

In [144]:
engineering_df.shape

(119390, 28)

In [ ]:
# Drop all rows containing Contract or Group customer_type
engineering_df = engineering_df[engineering_df['customer_type'] == 'Transient'].reset_index(drop=True)
engineering_df

,hotel,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,day_demand
0,Resort Hotel,0,342,27,0,0,2,0,0,1,Direct,Direct,0,0,0,3,No Deposit,0,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,122
1,Resort Hotel,0,737,27,0,0,2,0,0,1,Direct,Direct,0,0,0,4,No Deposit,0,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,122
2,Resort Hotel,0,7,27,0,1,1,0,0,1,Direct,Direct,0,0,0,0,No Deposit,0,0,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,122
3,Resort Hotel,0,13,27,0,1,1,0,0,1,Corporate,Corporate,0,0,0,0,No Deposit,1,0,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,122
4,Resort Hotel,0,14,27,0,2,2,0,0,1,Online TA,TA/TO,0,0,0,0,No Deposit,1,0,0,Transient,98.00,0,1,Check-Out,2015-07-03,2015-07-01,122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114732,City Hotel,0,23,35,2,5,2,0,0,1,Offline TA/TO,TA/TO,0,0,0,0,No Deposit,1,0,0,Transient,96.14,0,0,Check-Out,2017-09-06,2017-08-30,89
114733,City Hotel,0,102,35,2,5,3,0,0,1,Online TA,TA/TO,0,0,0,0,No Deposit,1,0,0,Transient,225.43,0,2,Check-Out,2017-09-07,2017-08-31,134
114734,City Hotel,0,34,35,2,5,2,0,0,1,Online TA,TA/TO,0,0,0,0,No Deposit,1,0,0,Transient,157.71,0,4,Check-Out,2017-09-07,2017-08-31,134
114735,City Hotel,0,109,35,2,5,2,0,0,1,Online TA,TA/TO,0,0,0,0,No Deposit,1,0,0,Transient,104.40,0,0,Check-Out,2017-09-07,2017-08-31,134


## <a id='toc3_2_'></a>[2.2 Length of Stay Feature Engineering](#toc0_)
- Calculate Length of Stay (los) feature

In [ ]:
engineering_df[['stays_in_week_nights', 'stays_in_weekend_nights']]

,stays_in_week_nights,stays_in_weekend_nights
0,0,0
1,0,0
2,1,0
3,1,0
4,2,0
...,...,...
114732,5,2
114733,5,2
114734,5,2
114735,5,2


In [147]:
# Add stays_in_weekend_nights + stays_in_week_nights
engineering_df['los'] = engineering_df['stays_in_week_nights'] + engineering_df['stays_in_weekend_nights']

In [148]:
# Sanity check
engineering_df['los'].value_counts().head(10)

los
2     26499
3     26457
1     20535
4     16960
7      7782
5      7574
6      3742
8      1119
10      920
9       782
Name: count, dtype: int64

Looking at the value counts for length of stay, over 90,000 of the 114,000 reservations stay between 1-4 nights.

## <a id='toc3_3_'></a>[2.3 Remove reservation_status](#toc0_)
Removing `reservation_status` due to the fact that this feature is highly correlated to our target variable `is_canceled` and that it shows what reservations are canceled.

In [ ]:
# Drop reservation_status column
engineering_df.drop(columns='reservation_status', inplace=True)

## <a id='toc3_4_'></a>[2.4 Remove highly correlated features: previous_cancellations](#toc0_)
- Through my initial modeling, I discovered that `previous_cancellations`had a very high coefficient and could contribute to early overfitting. Therefore, I removed this feature.

In [ ]:
# Dropping required_car_parking_spaces due to high coefficient
engineering_df.drop(columns='previous_cancellations', inplace=True)

In [ ]:
# Sanity check
engineering_df.head()

,hotel,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_bookings_not_canceled,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date,arrival_date,day_demand,los
0,Resort Hotel,0,342,27,0,0,2,0,0,1,Direct,Direct,0,0,3,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01,2015-07-01,122,0
1,Resort Hotel,0,737,27,0,0,2,0,0,1,Direct,Direct,0,0,4,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01,2015-07-01,122,0
2,Resort Hotel,0,7,27,0,1,1,0,0,1,Direct,Direct,0,0,0,No Deposit,0,0,0,Transient,75.0,0,0,2015-07-02,2015-07-01,122,1
3,Resort Hotel,0,13,27,0,1,1,0,0,1,Corporate,Corporate,0,0,0,No Deposit,1,0,0,Transient,75.0,0,0,2015-07-02,2015-07-01,122,1
4,Resort Hotel,0,14,27,0,2,2,0,0,1,Online TA,TA/TO,0,0,0,No Deposit,1,0,0,Transient,98.0,0,1,2015-07-03,2015-07-01,122,2


## <a id='toc3_5_'></a>[2.5 Extract features from datetime column: arrival_date](#toc0_)

Before we can scale, arrival_date and reservation_status_date needs to be transformed since a logistic regression model cannot take datetime

In [152]:
engineering_df.head(2)

,hotel,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_bookings_not_canceled,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date,arrival_date,day_demand,los
0,Resort Hotel,0,342,27,0,0,2,0,0,1,Direct,Direct,0,0,3,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01,2015-07-01,122,0
1,Resort Hotel,0,737,27,0,0,2,0,0,1,Direct,Direct,0,0,4,No Deposit,0,0,0,Transient,0.0,0,0,2015-07-01,2015-07-01,122,0


In [153]:
# Extract year, month, day of month from arrival_date 
engineering_df['arrival_year'] = engineering_df['arrival_date'].dt.year
engineering_df['arrival_month'] = engineering_df['arrival_date'].dt.month
engineering_df['arrival_dow'] = engineering_df['arrival_date'].dt.weekday
engineering_df['arrival_day'] = engineering_df['arrival_date'].dt.day


Not including Status_Date because of leakage when combined with arrival_date
Also removing date_diff because of leakage

In [154]:
# Drop original datetime columns for X_train
engineering_df.drop(columns='reservation_status_date', inplace=True)
engineering_df.drop(columns='arrival_date',inplace=True)


In [ ]:
# Sanity check 
engineering_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114737 entries, 0 to 114736
Data columns (total 29 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           114737 non-null  object 
 1   is_canceled                     114737 non-null  int64  
 2   lead_time                       114737 non-null  int64  
 3   arrival_date_week_number        114737 non-null  int64  
 4   stays_in_weekend_nights         114737 non-null  int64  
 5   stays_in_week_nights            114737 non-null  int64  
 6   adults                          114737 non-null  int64  
 7   children                        114737 non-null  int64  
 8   babies                          114737 non-null  int64  
 9   meal                            114737 non-null  int64  
 10  market_segment                  114737 non-null  object 
 11  distribution_channel            114737 non-null  object 
 12  is_repeated_gues

## <a id='toc3_6_'></a>[2.6 One Hot Encoding Feature Engineering:](#toc0_)
- hotel
- market_segment
- distribution_channel
- deposit_type
- customer_type


In [ ]:
# Instantiate OneHotEncoder
one_hot_encoder = OneHotEncoder(sparse_output=False)

In [157]:
# Fit and transform categorical features in OHE
encoded_array = one_hot_encoder.fit_transform(engineering_df[['hotel','market_segment', 'distribution_channel', 'deposit_type', 'customer_type']])
encoded_array

array([[0., 1., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 0., 1.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [1., 0., 0., ..., 0., 0., 1.],
       [1., 0., 0., ..., 0., 0., 1.]])

In [158]:
one_hot_encoder.categories_

[array(['City Hotel', 'Resort Hotel'], dtype=object),
 array(['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups',
        'Offline TA/TO', 'Online TA', 'Undefined'], dtype=object),
 array(['Corporate', 'Direct', 'GDS', 'TA/TO', 'Undefined'], dtype=object),
 array(['No Deposit', 'Non Refund', 'Refundable'], dtype=object),
 array(['Transient'], dtype=object)]

In [159]:
# Put into a dataframe to get column names
encoded_df = pd.DataFrame(encoded_array, columns=one_hot_encoder.get_feature_names_out(['hotel','market_segment', 'distribution_channel', 'deposit_type', 'customer_type']), dtype=int)

In [160]:
encoded_df.tail()

,hotel_City Hotel,hotel_Resort Hotel,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,distribution_channel_Corporate,distribution_channel_Direct,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Transient
114732,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,1
114733,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
114734,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
114735,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
114736,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1


In [ ]:
# Combine engineering_df and encoded_df for final_df for modeling
final_df = pd.concat([engineering_df,encoded_df],axis=1)

In [162]:
# Drop original columns that were used for One Hot Encoding
final_df.drop(columns='hotel',inplace=True)
final_df.drop(columns='market_segment',inplace=True)
final_df.drop(columns='distribution_channel',inplace=True)
final_df.drop(columns='deposit_type',inplace=True)
final_df.drop(columns='customer_type',inplace=True)

In [ ]:
# Final dataframe ready for modeling
final_df

,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,is_repeated_guest,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,day_demand,los,arrival_year,arrival_month,arrival_dow,arrival_day,hotel_City Hotel,hotel_Resort Hotel,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,distribution_channel_Corporate,distribution_channel_Direct,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Transient
0,0,342,27,0,0,2,0,0,1,0,0,3,0,0,0,0.00,0,0,122,0,2015,7,2,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
1,0,737,27,0,0,2,0,0,1,0,0,4,0,0,0,0.00,0,0,122,0,2015,7,2,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
2,0,7,27,0,1,1,0,0,1,0,0,0,0,0,0,75.00,0,0,122,1,2015,7,2,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
3,0,13,27,0,1,1,0,0,1,0,0,0,1,0,0,75.00,0,0,122,1,2015,7,2,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1
4,0,14,27,0,2,2,0,0,1,0,0,0,1,0,0,98.00,0,1,122,2,2015,7,2,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114732,0,23,35,2,5,2,0,0,1,0,0,0,1,0,0,96.14,0,0,89,7,2017,8,2,30,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,1
114733,0,102,35,2,5,3,0,0,1,0,0,0,1,0,0,225.43,0,2,134,7,2017,8,3,31,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
114734,0,34,35,2,5,2,0,0,1,0,0,0,1,0,0,157.71,0,4,134,7,2017,8,3,31,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
114735,0,109,35,2,5,2,0,0,1,0,0,0,1,0,0,104.40,0,0,134,7,2017,8,3,31,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1


# <a id='toc4_'></a>[3. Summary](#toc0_)

After initial modeling, I returned to feature engineering to correct leakage that caused overfitting due to reservation_status and reservation_status_date. Essentially, reservation_status contains data on if the reservation would ultimately be cancelled, no showed, or checked out.  Another discovery was when the arrival_date and reservation_status_date were both included, overfitting/leakage would occur because any reservations that cancelled would have a reservation status day BEFORE the arrival_date.  Whereas any reservation that checked out, would have the opposite, the reservation_status_date would be AFTER the arrival_date.  Both of these were addressed before proceeding to the next modeling phase.

A possible next iteration of feature engineering would be binning length of stay into smaller buckets. This could possibly reduce model complexity and overfitting as well.

# <a id='toc5_'></a>[4. Data Export](#toc0_)

In [ ]:
# Export final_df
final_df.to_csv('../data/processed/final_df.csv', index=False)

# Export engineered dataframe before OHE
engineering_df.to_csv('../data/processed/engineering_df.csv', index=False)